# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [9]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): https://github.com/hwarang97/jungle_1314_minigpt
GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ··········
이미 clone된 저장소를 사용합니다: /content/jungle_1314_minigpt
Repo: /content/jungle_1314_minigpt


In [10]:


# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [11]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

이미 존재합니다: /content/jungle_1314_minigpt/data/ratings_train.txt
이미 존재합니다: /content/jungle_1314_minigpt/data/ratings_test.txt
사전 학습 train 텍스트: /content/jungle_1314_minigpt/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /content/jungle_1314_minigpt/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /content/jungle_1314_minigpt/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /content/jungle_1314_minigpt/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /content/jungle_1314_minigpt/data/nsmc_sentiment_test.jsonl (49,997개)
LM train exists: True /content/jungle_1314_minigpt/data/nsmc_lm_train.txt
LM val exists: True /content/jungle_1314_minigpt/data/nsmc_lm_val.txt


In [12]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

train chars: 1379486
val chars: 120560
개재미없다. 감독의 연출력의 한계
이제서야 보게된 대 명작 연출미가 정말 훌륭하다!!!!!!!!
소주미라클을 만들어라
귀여운 캐릭터들도 많이 나와서 보러 가야 겠어요..
블랙 코미디가 싫어요.
평점깎고싶다10글자
TV시리즈가 너무재밌어서 영화는 기대안하고 봤는데 역시....최고네요
개인적 공감이 글쎄?
시작은 니시지마 때문에 봤는데 나름 괜찮은 영화 봤다고


## 2.5 Run Level Settings

Change only `RUN_LEVEL` to switch between quick Colab checks (`LIGHT`) and longer presentation runs (`BASIC`).


In [13]:
# Colab run-size settings: LIGHT is quick, BASIC is better for presentation logs.
RUN_LEVEL = "BASIC"  # "LIGHT" or "BASIC"

RUN_CONFIGS = {
    "LIGHT": {
        "CORPUS_LIMIT": 500_000,
        "VAL_CORPUS_LIMIT": 50_000,
        "VOCAB_SIZE": 2000,
        "CONTEXT_LENGTH": 64,
        "EMB_DIM": 128,
        "N_HEADS": 4,
        "N_LAYERS": 2,
        "BATCH_SIZE": 8,
        "NUM_EPOCHS": 2,
        "EVAL_FREQ": 100,
        "EVAL_ITER": 10,
        "CKPT_FREQ": 500,
        "SENTIMENT_TRAIN_LIMIT": 3_000,
        "SENTIMENT_VAL_LIMIT": 1_000,
        "SENTIMENT_TEST_LIMIT": 1_000,
        "FINETUNE_EPOCHS": 2,
    },
    "BASIC": {
        "CORPUS_LIMIT": 1_500_000,
        "VAL_CORPUS_LIMIT": 150_000,
        "VOCAB_SIZE": 3000,
        "CONTEXT_LENGTH": 128,
        "EMB_DIM": 192,
        "N_HEADS": 4,
        "N_LAYERS": 4,
        "BATCH_SIZE": 8,
        "NUM_EPOCHS": 2,
        "EVAL_FREQ": 100,
        "EVAL_ITER": 10,
        "CKPT_FREQ": 500,
        "SENTIMENT_TRAIN_LIMIT": 10_000,
        "SENTIMENT_VAL_LIMIT": 2_000,
        "SENTIMENT_TEST_LIMIT": 2_000,
        "FINETUNE_EPOCHS": 2,
    },
}

if RUN_LEVEL not in RUN_CONFIGS:
    raise ValueError(f"RUN_LEVEL must be one of {list(RUN_CONFIGS)}: {RUN_LEVEL}")

_cfg = RUN_CONFIGS[RUN_LEVEL]
CORPUS_LIMIT = min(_cfg["CORPUS_LIMIT"], len(corpus)) if corpus else _cfg["CORPUS_LIMIT"]
VAL_CORPUS_LIMIT = min(_cfg["VAL_CORPUS_LIMIT"], len(val_corpus)) if val_corpus else _cfg["VAL_CORPUS_LIMIT"]
VOCAB_SIZE = _cfg["VOCAB_SIZE"]
CONTEXT_LENGTH = _cfg["CONTEXT_LENGTH"]
EMB_DIM = _cfg["EMB_DIM"]
N_HEADS = _cfg["N_HEADS"]
N_LAYERS = _cfg["N_LAYERS"]
BATCH_SIZE = _cfg["BATCH_SIZE"]
NUM_EPOCHS = _cfg["NUM_EPOCHS"]
EVAL_FREQ = _cfg["EVAL_FREQ"]
EVAL_ITER = _cfg["EVAL_ITER"]
CKPT_FREQ = _cfg["CKPT_FREQ"]
SENTIMENT_TRAIN_LIMIT = _cfg["SENTIMENT_TRAIN_LIMIT"]
SENTIMENT_VAL_LIMIT = _cfg["SENTIMENT_VAL_LIMIT"]
SENTIMENT_TEST_LIMIT = _cfg["SENTIMENT_TEST_LIMIT"]
FINETUNE_EPOCHS = _cfg["FINETUNE_EPOCHS"]

PRETRAIN_LR = 3e-4
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 0.1


def make_gpt_config(drop_rate: float = 0.1) -> dict:
    return {
        "vocab_size": VOCAB_SIZE,
        "context_length": CONTEXT_LENGTH,
        "emb_dim": EMB_DIM,
        "n_heads": N_HEADS,
        "n_layers": N_LAYERS,
        "drop_rate": drop_rate,
        "qkv_bias": False,
    }


def get_or_train_tokenizer():
    from bpe import BPETokenizer

    if not corpus:
        raise ValueError("corpus is empty. Run the data preparation cells first.")

    vocab_path = repo_dir / "data" / f"vocab_bpe_{VOCAB_SIZE}.json"
    tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)
    if vocab_path.exists():
        tokenizer.load(vocab_path)
        print("BPE vocab loaded:", vocab_path)
    else:
        print(f"BPE vocab training: first {CORPUS_LIMIT:,} chars, vocab_size={VOCAB_SIZE}")
        tokenizer.train(corpus[:CORPUS_LIMIT])
        tokenizer.save(vocab_path)
        print("BPE vocab saved:", vocab_path)
    return tokenizer


print("RUN_LEVEL:", RUN_LEVEL)
print("pretrain:", {"chars": CORPUS_LIMIT, "vocab": VOCAB_SIZE, "context": CONTEXT_LENGTH, "batch": BATCH_SIZE, "epochs": NUM_EPOCHS})
print("model:", make_gpt_config())
print("sentiment limits:", SENTIMENT_TRAIN_LIMIT, SENTIMENT_VAL_LIMIT, SENTIMENT_TEST_LIMIT)


RUN_LEVEL: BASIC
pretrain: {'chars': 1379486, 'vocab': 3000, 'context': 128, 'batch': 8, 'epochs': 2}
model: {'vocab_size': 3000, 'context_length': 128, 'emb_dim': 192, 'n_heads': 4, 'n_layers': 4, 'drop_rate': 0.1, 'qkv_bias': False}
sentiment limits: 10000 2000 2000


## 2.6 Colab Actual Run Checklist

Use this notebook in two passes.

1. Quick check: set `RUN_LEVEL = "LIGHT"`, run the cells from top to bottom, and confirm the smoke checks pass.
2. Presentation run: set `RUN_LEVEL = "BASIC"`, then turn on the long training cells below.
3. Pretraining: in `## 7.5 Actual Pretraining Run`, set `RUN_ACTUAL_PRETRAIN = True`.
4. Fine-tuning: in `## 8.5 Actual Sentiment Fine-Tuning Run`, set `RUN_ACTUAL_FINETUNE = True`.
5. Logs to check:
   - `logs/pretrain_metrics.jsonl`: train loss and validation loss
   - `logs/pretrain_samples.jsonl`: generated samples
   - `logs/sentiment_metrics.jsonl`: train/validation/test loss and accuracy

Useful Colab checks:

```python
!tail -n 20 logs/pretrain_metrics.jsonl
!tail -n 5 logs/pretrain_samples.jsonl
!tail -n 20 logs/sentiment_metrics.jsonl
```


## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [14]:
run_pytest("tests/test_bpe.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_bpe.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 6 items

tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 16%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 33%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 50%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 66%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 83%]
tests/test_bpe.py::TestBPETrain::test_train_increases_vocab PASSED       [100%]

============================== 6 passed in 0.02s ===============================


선택한 테스트를 통과했습니다.


0

In [15]:
# Check BPE encode/decode with the current RUN_LEVEL tokenizer.
try:
    tokenizer = get_or_train_tokenizer()
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except (NotImplementedError, ValueError) as e:
    print("BPE is not ready or data is missing:", e)


BPE vocab training: first 1,379,486 chars, vocab_size=3000
BPE vocab saved: /content/jungle_1314_minigpt/data/vocab_bpe_3000.json
[2, 267, 1003, 532, 860, 1996, 36, 73, 114, 107, 112, 109, 119, 108, 505, 54, 55, 3]
이 영화는 정말 좋았다! English 123


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [16]:
run_pytest("tests/test_dataset.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_dataset.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 4 items

tests/test_dataset.py::TestGPTDataset::test_dataset_length PASSED        [ 25%]
tests/test_dataset.py::TestGPTDataset::test_dataset_getitem_shape PASSED [ 50%]
tests/test_dataset.py::TestCreateDataloader::test_dataloader_batch_shape PASSED [ 75%]
tests/test_dataset.py::TestInputEmbedding::test_input_embedding_shape PASSED [100%]

============================== 4 passed in 4.76s ===============================


선택한 테스트를 통과했습니다.


0

In [17]:
try:
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = get_or_train_tokenizer()
    token_ids = tokenizer.encode(corpus[:CORPUS_LIMIT])
    loader = create_dataloader(
        token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=True,
    )
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(
        vocab_size=VOCAB_SIZE,
        emb_dim=EMB_DIM,
        context_length=CONTEXT_LENGTH,
        drop_rate=0.1,
    )
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except (NotImplementedError, ValueError) as e:
    print("Dataset/Embedding is not ready or data is missing:", e)


BPE vocab loaded: /content/jungle_1314_minigpt/data/vocab_bpe_3000.json
torch.Size([8, 128]) torch.Size([8, 128]) torch.Size([8, 128, 192])


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [18]:
run_pytest("tests/test_attention.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_attention.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 2 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [ 50%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [100%]

============================== 2 passed in 1.65s ===============================


선택한 테스트를 통과했습니다.


0

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [19]:
run_pytest("tests/test_model.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_model.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 7 items

tests/test_model.py::TestLayerNorm::test_layernorm_shape PASSED          [ 14%]
tests/test_model.py::TestGELU::test_gelu_shape PASSED                    [ 28%]
tests/test_model.py::TestFeedForward::test_feedforward_shape PASSED      [ 42%]
tests/test_model.py::TestTransformerBlock::test_transformer_block_shape PASSED [ 57%]
tests/test_model.py::TestGPTModel::test_gpt_forward_shape PASSED         [ 71%]
tests/test_model.py::TestGPTModel::test_gpt_forward_with_targets_returns_loss PASSED [ 85%]
tests/test_model.py::TestGenerateTextSimple::test_generate_text_simple_shape PASSED [100%]

============================== 7 passed

0

In [20]:
try:
    import torch
    from model import GPTModel

    config = make_gpt_config(drop_rate=0.1)
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, min(16, config["context_length"])))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model is not ready:", e)


torch.Size([2, 16, 3000])


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [21]:
run_pytest("tests/test_train.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_train.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 5 items

tests/test_train.py::TestCalcLossBatch::test_calc_loss_batch_returns_scalar PASSED [ 20%]
tests/test_train.py::TestCalcLossLoader::test_calc_loss_loader_returns_float PASSED [ 40%]
tests/test_train.py::TestCheckpoint::test_save_load_checkpoint_restores_epoch_and_step PASSED [ 60%]
tests/test_train.py::TestGenerate::test_generate_shape PASSED            [ 80%]
tests/test_train.py::TestPlotLosses::test_plot_losses_callable PASSED    [100%]

============================== 5 passed in 6.88s ===============================


선택한 테스트를 통과했습니다.


0

In [22]:
# Run one loss/backward smoke test with the current RUN_LEVEL settings.
try:
    import torch
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = get_or_train_tokenizer()
    token_ids = tokenizer.encode(corpus[:CORPUS_LIMIT])
    loader = create_dataloader(
        token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=True,
    )
    inp, tgt = next(iter(loader))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GPTModel(make_gpt_config(drop_rate=0.1)).to(device)
    loss = calc_loss_batch(inp, tgt, model, device)
    loss.backward()
    print("device:", device)
    print("smoke loss:", loss.item())
except (NotImplementedError, ValueError) as e:
    print("Pretraining utilities are not ready or data is missing:", e)


BPE vocab loaded: /content/jungle_1314_minigpt/data/vocab_bpe_3000.json
device: cuda
smoke loss: 8.218156814575195


## 7.5 Actual Pretraining Run

Set `RUN_ACTUAL_PRETRAIN = True` to train with the selected `RUN_LEVEL` and write `logs/pretrain_metrics.jsonl` plus `logs/pretrain_samples.jsonl`.


In [23]:
RUN_ACTUAL_PRETRAIN = True  # Change to True when you want the longer run.

if RUN_ACTUAL_PRETRAIN:
    import torch
    from dataset import create_dataloader
    from model import GPTModel
    from train import train_model

    tokenizer = get_or_train_tokenizer()
    train_token_ids = tokenizer.encode(corpus[:CORPUS_LIMIT])
    val_text = val_corpus[:VAL_CORPUS_LIMIT] if val_corpus else corpus[:max(10_000, CORPUS_LIMIT // 10)]
    val_token_ids = tokenizer.encode(val_text)

    train_loader = create_dataloader(
        train_token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=True,
        drop_last=True,
    )
    val_loader = create_dataloader(
        val_token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GPTModel(make_gpt_config(drop_rate=0.1)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)

    train_losses = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        device,
        num_epochs=NUM_EPOCHS,
        eval_freq=EVAL_FREQ,
        eval_iter=EVAL_ITER,
        start_context="이 영화는",
        tokenizer=tokenizer,
        ckpt_freq=CKPT_FREQ,
        metrics_path=repo_dir / "logs" / "pretrain_metrics.jsonl",
        sample_path=repo_dir / "logs" / "pretrain_samples.jsonl",
    )

    print("pretrain losses:", train_losses)
    print("metrics:", repo_dir / "logs" / "pretrain_metrics.jsonl")
    print("samples:", repo_dir / "logs" / "pretrain_samples.jsonl")
else:
    print("Set RUN_ACTUAL_PRETRAIN = True and rerun this cell to start actual pretraining.")


BPE vocab loaded: /content/jungle_1314_minigpt/data/vocab_bpe_3000.json
step 100: train loss 7.3008, val loss 7.3155
step 200: train loss 7.2935, val loss 7.3117
step 300: train loss 7.3086, val loss 7.3072
step 400: train loss 7.2904, val loss 7.2957
step 500: train loss 7.2424, val loss 7.2548
step 600: train loss 7.1192, val loss 7.1455
step 700: train loss 6.9605, val loss 7.0039
이 영화는 나시하고 재미있다
지만 재미, 명작. 너무 봤음.. 한국는 아니다 그 
step 800: train loss 6.8213, val loss 6.8397
step 900: train loss 6.6109, val loss 6.6821
step 1000: train loss 6.5075, val loss 6.5380
step 1100: train loss 6.3198, val loss 6.4040
step 1200: train loss 6.2090, val loss 6.2928
step 1300: train loss 6.0737, val loss 6.1943
step 1400: train loss 5.9672, val loss 6.1196
step 1500: train loss 5.9102, val loss 6.0484
이 영화는 더 자성수에 이런 느낌
재미한다면 더 좋았는데...
이들 말하는영화
pretrain losses: [7.261629537771676, 6.409937240391894]
metrics: /content/jungle_1314_minigpt/logs/pretrain_metrics.jsonl
samples: /content/jungle_1314_minig

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [24]:
run_pytest("tests/test_finetune.py")

실행 명령: /usr/bin/python3 -m pytest tests/test_finetune.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 4 items

tests/test_finetune.py::TestMakeSentimentDataset::test_make_sentiment_dataset_splits_rows PASSED [ 25%]
tests/test_finetune.py::TestReviewSentimentDataset::test_review_sentiment_dataset_getitem PASSED [ 50%]
tests/test_finetune.py::TestGPTForSequenceClassification::test_sequence_classification_shape PASSED [ 75%]
tests/test_finetune.py::TestSentimentTrainEval::test_train_eval_functions_exist PASSED [100%]

============================== 4 passed in 1.61s ===============================


선택한 테스트를 통과했습니다.


0

## 8.5 Actual Sentiment Fine-Tuning Run

Set `RUN_ACTUAL_FINETUNE = True` to log train/validation/test loss and accuracy to `logs/sentiment_metrics.jsonl`.


In [25]:
RUN_ACTUAL_FINETUNE = True  # Change to True when you want actual fine-tuning.

if RUN_ACTUAL_FINETUNE:
    import json
    import torch
    from torch.utils.data import DataLoader
    from finetune import (
        GPTForSequenceClassification,
        ReviewSentimentDataset,
        evaluate_sentiment,
        train_epoch_sentiment,
    )
    from model import GPTModel

    def read_jsonl(path, limit=None):
        rows = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                rows.append(json.loads(line))
                if limit is not None and len(rows) >= limit:
                    break
        return rows

    tokenizer = globals().get("tokenizer") or get_or_train_tokenizer()
    if "model" not in globals():
        print("No pretrained model variable was found, so a new GPTModel is created.")
        model = GPTModel(make_gpt_config(drop_rate=0.1))

    train_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_train.jsonl", SENTIMENT_TRAIN_LIMIT)
    val_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_val.jsonl", SENTIMENT_VAL_LIMIT)
    test_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_test.jsonl", SENTIMENT_TEST_LIMIT)

    train_ds = ReviewSentimentDataset(train_data, tokenizer, max_length=CONTEXT_LENGTH)
    val_ds = ReviewSentimentDataset(val_data, tokenizer, max_length=CONTEXT_LENGTH)
    test_ds = ReviewSentimentDataset(test_data, tokenizer, max_length=CONTEXT_LENGTH)

    train_cls_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_cls_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_cls_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clf_model = GPTForSequenceClassification(model, num_labels=2).to(device)
    clf_optimizer = torch.optim.AdamW(clf_model.parameters(), lr=FINETUNE_LR)

    for epoch in range(FINETUNE_EPOCHS):
        train_loss, train_acc = train_epoch_sentiment(
            clf_model,
            train_cls_loader,
            clf_optimizer,
            device,
            epoch=epoch,
            metrics_path=repo_dir / "logs" / "sentiment_metrics.jsonl",
        )
        val_loss, val_acc = evaluate_sentiment(
            clf_model,
            val_cls_loader,
            device,
            split="val",
            epoch=epoch,
            metrics_path=repo_dir / "logs" / "sentiment_metrics.jsonl",
        )
        print(f"epoch {epoch}: train loss={train_loss:.4f}, train acc={train_acc:.4f}, val loss={val_loss:.4f}, val acc={val_acc:.4f}")

    test_loss, test_acc = evaluate_sentiment(
        clf_model,
        test_cls_loader,
        device,
        split="test",
        metrics_path=repo_dir / "logs" / "sentiment_metrics.jsonl",
    )
    print(f"test loss={test_loss:.4f}, test acc={test_acc:.4f}")
    print("metrics:", repo_dir / "logs" / "sentiment_metrics.jsonl")
else:
    print("Set RUN_ACTUAL_FINETUNE = True and rerun this cell to start actual fine-tuning.")


epoch 0: train loss=0.6812, train acc=0.5661, val loss=0.6214, val acc=0.6525
epoch 1: train loss=0.5766, train acc=0.6902, val loss=0.5715, val acc=0.6965
test loss=0.5466, test acc=0.7230
metrics: /content/jungle_1314_minigpt/logs/sentiment_metrics.jsonl


## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [26]:
run_pytest("tests/")

실행 명령: /usr/bin/python3 -m pytest tests/ -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/jungle_1314_minigpt
plugins: typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
collecting ... collected 28 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [  3%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [  7%]
tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 10%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 14%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 17%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 21%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 25%]
tests/test_bpe.py::TestBPETrain::test_train_in

0